# 3. Site economics: access, logistics and net revenue

Notebook 2 ranked the sites on wave power. This one asks whether that ranking
survives contact with the cost of operating there.

## The argument

A wave energy converter earns money in proportion to the sea state. It also
costs money to reach, and **it is hardest to reach in exactly the conditions
that earn the most**. Marine operations need a continuous weather window long
enough to transit out, work, and return; wave height decides whether a vessel
can work at all.

So gross revenue and access cost are positively correlated, and net revenue is
the difference between two quantities that move together. Which site wins is not
knowable from the resource alone.

Distance compounds it. A site 150 km offshore needs a window several hours
longer than one 30 km out for the same job — and long windows are
*disproportionately* rarer than short ones, because they need a run of calm
rather than a single calm hour.

## On the numbers below

Every cost is a **documented planning assumption**, not a quote. They live in
`src/economics/deployment.py` with the reasoning attached, and they are meant to
be right to within about a factor of two — enough to rank sites, which is what a
siting decision needs. Absolute values should not be quoted as a budget.

In [ ]:
# Run from anywhere in the repo: put the project root on the path.
import sys
from pathlib import Path

root = Path.cwd()
while root != root.parent and not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (11, 5)
print(f"project root: {root}")

In [ ]:
from src.config import NDBC_STATIONS, PILOT_STATIONS
from src.data.ndbc import load_station
from src.economics.accessibility import seasonal_accessibility, window_statistics
from src.economics.deployment import (
    AHTS, ANNUAL_OPERATIONS, CTV, MULTICAT,
    SiteLogistics, false_start_multiplier, window_hours_required,
)
from src.economics.revenue import (
    DeviceSpec, MarketSpec, annual_om_cost, evaluate_site,
)
from src.processing.wave_power_flux import ENERGY_PERIOD_FACTORS, calculate_wave_power_flux

YEARS = list(range(2015, 2024))
records = {s: load_station(s, YEARS) for s in PILOT_STATIONS}
flux = {
    s: calculate_wave_power_flux(df["WVHT"], df["APD"] * ENERGY_PERIOD_FACTORS["VTM02"])
    for s, df in records.items()
}

## Distance to port

Approximate steaming distance from the assumed support base to each buoy.
These are straight-line estimates and should be replaced with real routed
distances before anything is committed.

In [ ]:
# Grays Harbor, WA as the support port.
PORT_LAT, PORT_LON = 46.90, -124.10

def rough_distance_km(lat, lon, lat0=PORT_LAT, lon0=PORT_LON):
    """Equirectangular approximation - adequate at these separations."""
    mean_lat = np.radians((lat + lat0) / 2)
    dx = np.radians(lon - lon0) * np.cos(mean_lat) * 6371.0
    dy = np.radians(lat - lat0) * 6371.0
    return float(np.hypot(dx, dy))

sites = {}
for station_id in PILOT_STATIONS:
    info = NDBC_STATIONS[station_id]
    distance = rough_distance_km(info.latitude, info.longitude)
    sites[station_id] = SiteLogistics(
        distance_km=distance, water_depth_m=info.depth_m or 120.0
    )
    print(f"{station_id} {info.name:<28} {distance:6.1f} km   depth {sites[station_id].water_depth_m:.0f} m")

## Vessels and what they need

Three vessel classes, each with a working wave-height limit. That limit is the
coupling between the wave climate and the cost model.

In [ ]:
pd.DataFrame([
    {"vessel": v.name, "day_rate_usd": v.day_rate_usd, "speed_kn": v.transit_speed_kn,
     "max_Hs_work_m": v.max_hs_operate_m, "mobilisation_usd": v.mobilisation_usd}
    for v in (AHTS, MULTICAT, CTV)
])

In [ ]:
# Required window grows with distance - the first of two penalties.
distances = np.array([20, 50, 100, 150, 250])
rows = []
for operation in ANNUAL_OPERATIONS:
    for d in distances:
        rows.append({"operation": operation.name, "distance_km": d,
                     "window_hours": window_hours_required(operation, d)})
pd.DataFrame(rows).pivot(index="operation", columns="distance_km", values="window_hours").round(1)

## Accessibility: how often is that window available?

The second penalty, and the non-linear one.

In [ ]:
access = []
for station_id, df in records.items():
    for operation in ANNUAL_OPERATIONS:
        required = window_hours_required(operation, sites[station_id].distance_km)
        stats = window_statistics(df["WVHT"], operation.vessel.max_hs_operate_m, required)
        access.append({
            "station": station_id, "operation": operation.name,
            "limit_m": operation.vessel.max_hs_operate_m,
            "window_needed_h": round(required, 1),
            "accessible_%": round(100 * stats["accessible_fraction"], 1),
            "windows_per_year": round(stats["windows_per_year"], 1),
            "expected_wait_h": round(stats["expected_wait_hours"], 0),
        })
pd.DataFrame(access)

Compare `accessible_%` with `windows_per_year`. A site can be below the working
limit 40% of the time and still offer very few *usable* windows, because the
calm hours are scattered rather than contiguous. The naive accessibility
percentage is the number people quote; the window count is the one that governs
operations.

### Seasonality is the operationally decisive fact

In [ ]:
operation = [op for op in ANNUAL_OPERATIONS if "Unscheduled" in op.name][0]
for station_id, df in records.items():
    required = window_hours_required(operation, sites[station_id].distance_km)
    table = seasonal_accessibility(df["WVHT"], operation.vessel.max_hs_operate_m, required)
    print(f"\n=== {station_id} {NDBC_STATIONS[station_id].name} "
          f"(needs {required:.0f} h below {operation.vessel.max_hs_operate_m} m) ===")
    print(table[["accessible_fraction", "windows_per_year", "expected_wait_hours"]].round(2).to_string())

Winter is both the energetic season and the inaccessible one. A failure in
December is a fundamentally different proposition from one in July, and any
maintenance plan built on the annual average will be wrong exactly when it
matters most.

## Net revenue

Bringing resource, access and logistics together. `gross_annual_usd` is what a
resource-only study reports; `net_annual_usd` is what is left after energy
losses, downtime, O&M and annualised capital.

In [ ]:
device = DeviceSpec()
market = MarketSpec()
print(f"{device.name}: capture width {device.capture_width_m} m, "
      f"efficiency {device.efficiency:.0%}, rated {device.rated_power_kw:.0f} kW, "
      f"survival cut-out {device.survival_hs_m} m")
print(f"Market: ${market.energy_price_usd_per_mwh}/MWh")

results = {}
for station_id in PILOT_STATIONS:
    results[station_id] = evaluate_site(
        records[station_id]["WVHT"], flux[station_id], sites[station_id], device, market
    )

economics = pd.DataFrame(results).T
economics.insert(0, "name", [NDBC_STATIONS[s].name for s in economics.index])
economics[[
    "name", "distance_km", "mean_flux_kw_per_m", "capacity_factor",
    "clipped_fraction", "survival_fraction", "downtime_fraction",
]].round(3)

In [ ]:
money = economics[[
    "gross_annual_usd", "delivered_annual_usd", "om_annual_usd",
    "annualised_capex_usd", "net_annual_usd",
]] / 1000.0
money.columns = [c.replace("_usd", "_$k") for c in money.columns]
money.round(0)

### Does the ranking change?

In [ ]:
ranking = pd.DataFrame({
    "resource_rank": economics["mean_flux_kw_per_m"].rank(ascending=False).astype(int),
    "gross_rank": economics["gross_annual_usd"].rank(ascending=False).astype(int),
    "net_rank": economics["net_annual_usd"].rank(ascending=False).astype(int),
    "mean_flux_kW_per_m": economics["mean_flux_kw_per_m"].round(1),
    "net_annual_$k": (economics["net_annual_usd"] / 1000).round(0),
    "downtime_%": (100 * economics["downtime_fraction"]).round(1),
})
ranking

## Sensitivity

The absolute numbers depend on assumptions that are uncertain by a factor of
two or more. What matters is whether the *ranking* is stable when they move.

In [ ]:
def rank_under(price, capture_width, day_rate_scale, unscheduled_scale):
    """Re-rank sites under altered assumptions."""
    from dataclasses import replace
    import src.economics.deployment as dep

    original = dep.ANNUAL_OPERATIONS
    scaled_vessels = {}
    def scale(v):
        if v.name not in scaled_vessels:
            scaled_vessels[v.name] = replace(v, day_rate_usd=v.day_rate_usd * day_rate_scale)
        return scaled_vessels[v.name]

    dep.ANNUAL_OPERATIONS = tuple(
        replace(op, vessel=scale(op.vessel),
                per_year=op.per_year * (unscheduled_scale if "Unscheduled" in op.name else 1.0))
        for op in original
    )
    try:
        nets = {}
        for s in PILOT_STATIONS:
            r = evaluate_site(
                records[s]["WVHT"], flux[s], sites[s],
                DeviceSpec(capture_width_m=capture_width),
                MarketSpec(energy_price_usd_per_mwh=price),
            )
            nets[s] = r["net_annual_usd"] / 1000.0
    finally:
        dep.ANNUAL_OPERATIONS = original
    return nets

scenarios = {
    "baseline":            dict(price=120, capture_width=20, day_rate_scale=1.0, unscheduled_scale=1.0),
    "low price ($60)":     dict(price=60,  capture_width=20, day_rate_scale=1.0, unscheduled_scale=1.0),
    "high price ($250)":   dict(price=250, capture_width=20, day_rate_scale=1.0, unscheduled_scale=1.0),
    "narrow capture (10m)":dict(price=120, capture_width=10, day_rate_scale=1.0, unscheduled_scale=1.0),
    "wide capture (35m)":  dict(price=120, capture_width=35, day_rate_scale=1.0, unscheduled_scale=1.0),
    "vessels 2x dearer":   dict(price=120, capture_width=20, day_rate_scale=2.0, unscheduled_scale=1.0),
    "3x failure rate":     dict(price=120, capture_width=20, day_rate_scale=1.0, unscheduled_scale=3.0),
}
sens = pd.DataFrame({k: rank_under(**v) for k, v in scenarios.items()}).T
sens["best_site"] = sens[list(PILOT_STATIONS)].idxmax(axis=1)
sens.round(0)

If `best_site` is the same across every row, the ranking is robust to the
assumptions and can be acted on even though the absolute figures cannot. If it
flips, the decision genuinely depends on a number we do not know, and that
number is what to go and find out.

## Where the forecast has a dollar value

This is where the forecasting work reconnects to the business.

Vessels are not kept on hire for weeks waiting for weather — they are
**mobilised against a forecast**. If the forecast is wrong the vessel sails,
aborts, and the day is paid for anyway. So forecast skill converts directly into
avoided false starts.

GEFS measures 0.41 m RMSE at +24 h at 46041 (see `docs/track_b_results.md`).
Below is what that is worth, and what a better or worse forecast would be worth.

In [ ]:
rows = []
for rmse in (0.0, 0.2, 0.41, 0.6, 1.0):
    multiplier = false_start_multiplier(rmse, MULTICAT.max_hs_operate_m)
    nets = {}
    for s in PILOT_STATIONS:
        om = annual_om_cost(records[s]["WVHT"], sites[s], forecast_rmse_m=rmse)
        nets[s] = om["total_annual_usd"] / 1000.0
    rows.append({"forecast_RMSE_m": rmse, "sailings_per_job": round(multiplier, 2), **{f"{s}_OM_$k": round(v) for s, v in nets.items()}})
pd.DataFrame(rows)

Read the first column against the last. The gap between a perfect forecast and
the one we have is the annual value of forecast skill at these sites — and the
gap between GEFS and a *worse* forecast is what the physics model is already
saving.

That is the honest framing of the forecasting work in this repository: its value
is in **scheduling marine operations**, not in predicting energy output. Energy
revenue depends on the long-run climate, which is a statistics problem. Vessel
dispatch depends on the next 24–48 hours, which is a forecasting problem — and
one GEFS already solves about twice as well as anything built here from buoy
history alone.

## Caveats

1. Distances are straight-line, not routed, and assume a single support port.
2. Vessel rates, failure rates and durations are planning assumptions.
3. The device specification is a placeholder for a 61 m structure; capture width
   and rated power are the most influential unknowns, as the sensitivity table
   shows.
4. Three buoy sites are not a spatial survey. Extending this to the gridded
   Copernicus field would give a continuous map — the code takes any Hs and flux
   series, so the same functions apply per grid cell.
5. No consenting, cabling, grid-connection or insurance costs. For a site far
   offshore, export cabling would likely dominate everything modelled here.